In [ ]:
!nvidia-smi

Tue Jun  2 18:06:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q transformers datasets accelerate
!pip install -q torch
!pip install -q peft
!pip install -q einops
!pip install -q tqdm

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "EleutherAI/pythia-160m"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    output_hidden_states=True
)

model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/569 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/375M [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPTNeoXForCausalLM(
  (gpt_neox): GPTNeoXModel(
    (embed_in): Embedding(50304, 768)
    (emb_dropout): Dropout(p=0.0, inplace=False)
    (layers): ModuleList(
      (0-11): 12 x GPTNeoXLayer(
        (input_layernorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (post_attention_layernorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (post_attention_dropout): Dropout(p=0.0, inplace=False)
        (post_mlp_dropout): Dropout(p=0.0, inplace=False)
        (attention): GPTNeoXAttention(
          (query_key_value): Linear(in_features=768, out_features=2304, bias=True)
          (dense): Linear(in_features=768, out_features=768, bias=True)
        )
        (mlp): GPTNeoXMLP(
          (dense_h_to_4h): Linear(in_features=768, out_features=3072, bias=True)
          (dense_4h_to_h): Linear(in_features=3072, out_features=768, bias=True)
          (act): GELUActivation()
        )
      )
    )
    (final_layer_norm): LayerNorm((768,), eps=1e-05, elementwi

In [ ]:
print(model)

GPTNeoXForCausalLM(
  (gpt_neox): GPTNeoXModel(
    (embed_in): Embedding(50304, 768)
    (emb_dropout): Dropout(p=0.0, inplace=False)
    (layers): ModuleList(
      (0-11): 12 x GPTNeoXLayer(
        (input_layernorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (post_attention_layernorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (post_attention_dropout): Dropout(p=0.0, inplace=False)
        (post_mlp_dropout): Dropout(p=0.0, inplace=False)
        (attention): GPTNeoXAttention(
          (query_key_value): Linear(in_features=768, out_features=2304, bias=True)
          (dense): Linear(in_features=768, out_features=768, bias=True)
        )
        (mlp): GPTNeoXMLP(
          (dense_h_to_4h): Linear(in_features=768, out_features=3072, bias=True)
          (dense_4h_to_h): Linear(in_features=3072, out_features=768, bias=True)
          (act): GELUActivation()
        )
      )
    )
    (final_layer_norm): LayerNorm((768,), eps=1e-05, elementwi

In [ ]:
print(len(model.gpt_neox.layers))

12


In [ ]:
import torch

text = "Python functions are useful for machine learning."

inputs = tokenizer(
    text,
    return_tensors="pt"
)

with torch.no_grad():
    outputs = model(**inputs)

hidden_states = outputs.hidden_states

print(len(hidden_states))

13


In [ ]:
layer6 = hidden_states[6]

print(layer6.shape)

torch.Size([1, 8, 768])


In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "Salesforce/wikitext",
    "wikitext-2-raw-v1",
    split="train"
)

print(dataset)

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 36718
})


In [ ]:
print(dataset)

Dataset({
    features: ['text'],
    num_rows: 36718
})


In [ ]:
print(dataset[0])

{'text': ''}


In [ ]:
import torch
from tqdm import tqdm

def collect_activations(
    dataset,
    model,
    tokenizer,
    target_layer=6,
    num_samples=5000,
    max_length=128
):

    activations = []

    model.eval()

    for i in tqdm(range(num_samples)):

        text = dataset[i]["text"]

        if len(text.strip()) == 0:
            continue

        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=max_length
        )

        inputs = {
            k: v.cuda()
            for k, v in inputs.items()
        }

        with torch.no_grad():

            outputs = model(**inputs)

        layer = outputs.hidden_states[target_layer]

        layer = layer.squeeze(0)

        activations.append(
            layer.cpu()
        )

    return torch.cat(
        activations,
        dim=0
    )

In [ ]:
device = "cuda"

model = model.to(device)

In [ ]:
print(next(model.parameters()).device)

cuda:0


In [ ]:
activations = collect_activations(
    dataset,
    model,
    tokenizer,
    target_layer=6,
    num_samples=5000
)

print(activations.shape)

100%|██████████| 5000/5000 [00:48<00:00, 104.14it/s]


torch.Size([225843, 768])


In [ ]:
print(activations[0][:10])

tensor([ 1.4580, -2.0215,  1.3545, -0.3782, -1.8594,  0.5898,  1.2500,  0.2524,
        -0.9355, -0.4460], dtype=torch.float16)


In [ ]:
torch.save(
    activations,
    "/content/drive/MyDrive/layer6_activations.pt"
)

In [ ]:
import os

size_mb = os.path.getsize(
    "/content/drive/MyDrive/layer6_activations.pt"
)/(1024**2)

print(size_mb)

330.82628440856934


In [ ]:
loaded = torch.load(
    "/content/drive/MyDrive/layer6_activations.pt"
)

print(loaded.shape)

torch.Size([225843, 768])


In [ ]:
from torch.utils.data import (
    TensorDataset,
    DataLoader
)

dataset = TensorDataset(
    activations.float()
)

loader = DataLoader(
    dataset,
    batch_size=512,
    shuffle=True
)

In [ ]:
import torch
import torch.nn as nn

In [ ]:
class SparseAutoencoder(nn.Module):

    def __init__(
        self,
        input_dim=768,
        dict_size=6144,
        k=30
    ):
        super().__init__()

        self.k = k

        self.encoder = nn.Linear(
            input_dim,
            dict_size
        )

        self.decoder = nn.Linear(
            dict_size,
            input_dim,
            bias=False
        )

        self.pre_bias = nn.Parameter(
            torch.zeros(input_dim)
        )

    @torch.no_grad()
    def normalize_decoder(self):
        norms = self.decoder.weight.data.norm(
            dim=0, keepdim=True
        )
        self.decoder.weight.data = (
            self.decoder.weight.data
            / norms.clamp(min=1e-8)
        )

    def forward(self, x):

        x_centered = x - self.pre_bias

        pre_activations = self.encoder(x_centered)


        topk_vals, topk_idx = torch.topk(
            pre_activations, self.k, dim=-1
        )

        features = torch.zeros_like(pre_activations)
        features.scatter_(-1, topk_idx, torch.relu(topk_vals))

        reconstruction = (
            self.decoder(features)
            + self.pre_bias
        )

        return reconstruction, features

In [ ]:
device = "cuda"

sae = SparseAutoencoder(
    input_dim=768,
    dict_size=6144
).to(device)

In [ ]:
def sae_loss(
    x,
    reconstruction,
    features,
    sparsity_weight=0

    recon_loss = (
        (x - reconstruction) ** 2
    ).mean()

    total_loss = recon_loss

    return (
        total_loss,
        recon_loss,
        torch.tensor(0.0)
    )

In [ ]:
activations = activations.float()

activations = activations.to(device)

In [ ]:
sae = SparseAutoencoder(
    input_dim=768,
    dict_size=6144,
    k=30
).to(device)

optimizer = torch.optim.Adam(
    sae.parameters(),
    lr=1e-4
)

scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer,
    start_factor=0.1,
    end_factor=1.0,
    total_iters=500
)

In [ ]:
epochs = 10
global_step = 0

for epoch in range(epochs):

    total_loss = 0
    total_recon = 0

    for batch in loader:

        x = batch[0].to(device)

        reconstruction, features = sae(x)

        loss, recon_loss, _ = sae_loss(x, reconstruction, features)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        sae.normalize_decoder()
        scheduler.step()

        global_step += 1

        total_loss += loss.item()
        total_recon += recon_loss.item()

    print(
        f"Epoch {epoch+1} | "
        f"Loss={total_loss/len(loader):.4f} | "
        f"Recon={total_recon/len(loader):.4f}"
    )

Epoch 1 | Loss=0.0785 | Recon=0.0785
Epoch 2 | Loss=0.0759 | Recon=0.0759
Epoch 3 | Loss=0.0737 | Recon=0.0737
Epoch 4 | Loss=0.0718 | Recon=0.0718
Epoch 5 | Loss=0.0700 | Recon=0.0700
Epoch 6 | Loss=0.0686 | Recon=0.0686
Epoch 7 | Loss=0.0712 | Recon=0.0712
Epoch 8 | Loss=0.0667 | Recon=0.0667
Epoch 9 | Loss=0.0652 | Recon=0.0652
Epoch 10 | Loss=0.0642 | Recon=0.0642


In [ ]:
with torch.no_grad():
    sample = activations[:1000].float().to(device)
    _, features = sae(sample)
    active_fraction = (features > 0).float().mean()
    print("Active Fraction:", active_fraction.item())
    print("Expected:", 30/6144)  # HAD TRAINED SAE AGAIN ON 10 EPOCHS TO CONVERGE LOSS

Active Fraction: 0.0048828125
Expected: 0.0048828125


In [ ]:
torch.save(
    sae.state_dict(),
    "/content/drive/MyDrive/sae_layer6_base.pt"
)
print("Saved.")

Saved.
